In [1]:

import pandas as pd
import os
from bokeh.plotting import figure, save, output_file, show
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, HoverTool, Legend
from bokeh.palettes import Spectral11
import numpy as np

def create_cdf_plots(folder_path):
    line_styles = ['solid', 'dashed', 'dotted', 'dotdash', 'dashdot']
    
    p = figure(title='Cumulative Distribution of Ranks',
               x_axis_label='Rank',
               y_axis_label='Percentage of Items (%)',
               width=1500, height=1000)

    # Add prior rank CDF first
    first_file = [f for f in os.listdir(folder_path) if f.endswith('.csv')][0]
    df = pd.read_csv(os.path.join(folder_path, first_file))
    prior_ranks = sorted(df['prior_rank'].values)
    x_vals = np.arange(0, 201)
    y_prior_cdf = [sum(1 for r in prior_ranks if r <= x) / len(prior_ranks) * 100 for x in x_vals]
    
    source_prior = ColumnDataSource(data={
        'x': x_vals,
        'y': y_prior_cdf,
        'name': ['Prior Rank'] * len(x_vals)
    })
    
    p.line('x', 'y', 
           line_width=2,
           line_color='black',
           line_dash='dashed',
           source=source_prior,
           legend_label='Prior Rank')
    
    # Process each file for current ranks
    for i, filename in enumerate(os.listdir(folder_path)):
        if filename.endswith('.csv'):
            df = pd.read_csv(os.path.join(folder_path, filename))
            ranks = sorted(df['rank'].values)
            y_cdf = [sum(1 for r in ranks if r <= x) / len(ranks) * 100 for x in x_vals]
            
            source = ColumnDataSource(data={
                'x': x_vals,
                'y': y_cdf,
                'avg_rank': [df['rank'].mean()] * len(x_vals),
                'median_rank': [df['rank'].median()] * len(x_vals),
                'avg_time': [df['totaltime'].mean()] * len(x_vals),
                'name': [filename] * len(x_vals)
            })
            
            p.line('x', 'y', 
                  line_width=2,
                  line_color=Spectral11[i % 11],
                  line_dash=line_styles[i % len(line_styles)],
                  source=source,
                  legend_label=filename)
    
    hover = HoverTool(tooltips=[
        ('File', '@name'),
        ('Rank', '@x'),
        ('Percentage', '@y{0.0}%'),
        ('Avg Rank', '@avg_rank{0.0}'),
        ('Median Rank', '@median_rank{0.0}'),
        ('Avg Time', '@avg_time{0.00}s')
    ])
    p.add_tools(hover)
    
    p.legend.location = "bottom_right"
    p.legend.click_policy = "hide"
    
    output_file("rank_distributions.html")
    show(p)

folder_path = 'jan1_results'
create_cdf_plots(folder_path)
